In [ ]:
!pip install nltk

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
import re


from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.tokenize import word_tokenize


from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder,MultiLabelBinarizer
from sklearn.metrics import classification_report,hamming_loss


from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding,LSTM,Dense,Dropout,Bidirectional,GRU,SimpleRNN
from tensorflow.keras.callbacks import EarlyStopping# short cut for model perfomance
from tensorflow.keras.utils import to_categorical

nltk.download('stopwords',quiet=True)
nltk.download('wordnet',quiet=True)
nltk.download('punkt_tab',quiet=True)
nltk.download('punkt',quiet=True)


In [ ]:
import pandas as pd

train_df = pd.read_csv("/content/train.csv")


train_df.head()


In [ ]:
# Basic exploration
print(train_df.head())
print(train_df.info())
print(train_df.describe())

In [ ]:
print(train_df.isnull().sum())

In [ ]:
train_df['toxic'].value_counts()

Clean Text

In [ ]:
#clean_text is the preprocessed version of the original text, where noise is removed and words are normalized for model training.
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z ]', '', text)
    return text

train_df['clean_text'] = train_df['comment_text'].apply(clean_text)

4. Input & Target

In [ ]:
X = train_df['clean_text']

y = train_df[['toxic','severe_toxic','obscene','threat','insult','identity_hate']]

5.Tokenize and Padding

In [ ]:
tokenizer = Tokenizer(num_words=5000)
tokenizer.fit_on_texts(X)

seq = tokenizer.texts_to_sequences(X)

Max_len = 100
X_pad = pad_sequences(seq, maxlen=Max_len, padding='post')

In [ ]:
tokenizer.index_word # based on word frenqunce give numbers

In [ ]:
X.shape

6. Train-Test Split

In [ ]:
X_Train,X_Test,y_Train,y_Test=train_test_split(X_pad,y,test_size=0.2)
X_Train.shape,X_Test.shape,y_Train.shape,y_Test.shape

LSTM model (RNN-based)

In [ ]:
model = Sequential([
    Embedding(input_dim=5000, output_dim=64, input_length=Max_len),
    Bidirectional(LSTM(128)),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dropout(0.2),
    Dense(6, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

9. Train

In [ ]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=2,
    restore_best_weights=True
)

history = model.fit(
    X_Train, y_Train,
    validation_data=(X_Test, y_Test),
    epochs=10,
    batch_size=64,
    callbacks=[early_stop]
)

In [ ]:
import pickle

# Save the Keras model
model.save('toxic_model.h5')

# Save the tokenizer
with open('tokenizer.pkl', 'wb') as handle:
    pickle.dump(tokenizer, handle, protocol=pickle.HIGHEST_PROTOCOL)

print("Model and tokenizer saved successfully!")

In [ ]:
# Generate predictions before attempting to display them
Y_train_pred = (model.predict(X_Train, verbose=0) >= 0.3).astype(int)
Y_train_pred

In [ ]:
# Hamming Loss
from sklearn.metrics import hamming_loss

Y_train_pred = (model.predict(X_Train, verbose=0) >= 0.5).astype(int)
Y_test_pred = (model.predict(X_Test, verbose=0) >= 0.5).astype(int)

train_hamming = hamming_loss(y_Train, Y_train_pred)
test_hamming = hamming_loss(y_Test, Y_test_pred)

print(f"Train Hamming Loss: {train_hamming}")#It measures how many labels are predicted incorrectly, averaged over all labels and samples.
print(f"Test Hamming Loss: {test_hamming}")

In [ ]:
y_pred = model.predict(X_Test)
y_pred = (y_pred > 0.2).astype(int)

In [ ]:
from sklearn.metrics import classification_report

labels = ['toxic','severe_toxic','obscene','threat','insult','identity_hate']

print(classification_report(y_Test, y_pred, target_names=labels))

In [ ]:
def predict_Topic(review):
    seq=tokenizer.texts_to_sequences([clean_text(review)])
    padded=pad_sequences(seq,maxlen=200,padding='post')
    pred_prob=topic_model.predict(padded,verbose=0)[0]
    topics=[mlb.classes_[i] for i,p in enumerate(pred_prob) if p>=0.5]

    return topics

In [ ]:
def predict_toxic(text):
    # Preprocess
    cleaned = clean_text(text)
    seq = tokenizer.texts_to_sequences([cleaned])
    padded = pad_sequences(seq, maxlen=Max_len, padding='post')

    # Predict
    pred_prob = model.predict(padded, verbose=0)[0]

    # Results
    results = {labels[i]: round(float(pred_prob[i]), 4) for i in range(len(labels)) if pred_prob[i] >= 0.2}
    return results if results else "Clean/Neutral"

# Alias for compatibility
predict_sentiment = predict_toxic

print(f"Result 1: {predict_toxic('Awsome Phone! Battery life is Amazing and Camera quality is superb')}")
print(f"Result 2: {predict_toxic('Worst product ever. stopped working after 2 days')}")
print(f"Result 3: {predict_sentiment('Average Phone. Nothing special about it')}")

In [ ]:
predict_sentiment('''very good.''')

In [ ]:
predict_sentiment("You are stupid")

In [ ]:
predict_sentiment("I have been watching your behavior for a while and honestly you are one of the most annoying and useless people I have ever seen. You keep making mistakes and acting like you know everything, but in reality you are just embarrassing yourself. If you continue like this, people will stop respecting you completely.")

In [ ]:
predict_sentiment('''Very good mobile with great battery backup.''')

In [ ]:
LSTM model (RNN-based)

Transformer-based model training (BERT/RoBERTa)

In [ ]:
# Since sentiment_model_hf is a pipeline, we save its underlying model and tokenizer
save_path = './roberta_toxic_model'
sentiment_model_hf.model.save_pretrained(save_path)
sentiment_model_hf.tokenizer.save_pretrained(save_path)

print(f"RoBERTa model and tokenizer saved successfully to {save_path}!")

In [ ]:
from transformers import pipeline

In [ ]:
X_Train

In [ ]:
Y_train_pred

In [ ]:
X_Train,X_Test,y_Train,y_Test=train_test_split(X_pad,y,test_size=0.2)
X_Train.shape,X_Test.shape,y_Train.shape,y_Test.shape

In [ ]:
sentiment_model_hf=pipeline('sentiment-analysis',model='cardiffnlp/twitter-roberta-base-sentiment-latest')
sentiment_model_hf

In [ ]:
from transformers import pipeline

# Re-initialize the pipeline using the imported function
sentiment_model_hf = pipeline('sentiment-analysis', model='cardiffnlp/twitter-roberta-base-sentiment-latest')

# Call the model correctly
result = sentiment_model_hf('very very bad')
print(result)

In [ ]:
df_Sample=train_df.sample(10)
df_Sample

In [ ]:
df_Sample['Sentiment_Prediction']=df_Sample['clean_text'].apply(lambda x:sentiment_model_hf(x)[0]['label'].title())
df_Sample['Sentiment_Score']=df_Sample['clean_text'].apply(lambda x:sentiment_model_hf(x)[0]['score'])
df_Sample

In [ ]:
stremlit

In [ ]:
%%writefile streamlit_app.py
import streamlit as st
import pandas as pd
import numpy as np
import re
from transformers import pipeline

# Page configuration
st.set_page_config(page_title="Toxicity AI Dashboard", layout="wide")

# Custom CSS for 4D Animated Background
st.markdown("""
<style>
@keyframes gradient {
    0% { background-position: 0% 50%; }
    50% { background-position: 100% 50%; }
    100% { background-position: 0% 50%; }
}
.stApp {
    background: linear-gradient(-45deg, #ee7752, #e73c7e, #23a6d5, #23d5ab);
    background-size: 400% 400%;
    animation: gradient 15s ease infinite;
    color: white;
}
.stMarkdown, .stHeader, p, h1, h2, h3, label {
    color: white !important;
}
.stButton>button {
    background-color: #ffffff33;
    color: white;
    border-radius: 20px;
    border: 1px solid white;
}
</style>
""", unsafe_allow_html=True)

@st.cache_resource
def load_model():
    return pipeline('sentiment-analysis', model='cardiffnlp/twitter-roberta-base-sentiment-latest')

sentiment_model = load_model()

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z ]', '', text)
    return text

st.title("✨ Toxicity Prediction Dashboard")

tab1, tab2 = st.tabs(["Real-time Analysis", "Bulk Insights"])

with tab1:
    st.header("Predict Toxicity & Sentiment")
    user_input = st.text_area("Enter comment for deep analysis:", placeholder="Type something here...")

    if st.button("Run AI Analysis"):
        if user_input:
            cleaned = clean_text(user_input)
            result = sentiment_model(cleaned)[0]
            label = result['label'].upper()
            score = result['score']

            # Visual Score Metric
            st.subheader(f"Result: {label}")
            st.progress(score)
            st.write(f"Confidence Level: {round(score*100, 2)}%")

            if label == 'NEGATIVE':
                st.error("⚠️ Potential Toxicity Detected")
            elif label == 'POSITIVE':
                st.success("✅ Friendly/Positive Content")
            else:
                st.info("ℹ️ Neutral Content")
        else:
            st.warning("Please enter some text first.")

with tab2:
    st.header("Data Exploration")
    st.write("Upload a CSV to process multiple comments at once.")
    uploaded_file = st.file_uploader("Choose a CSV file", type="csv")
    if uploaded_file:
        df_bulk = pd.read_csv(uploaded_file)
        st.dataframe(df_bulk.head(10))


In [ ]:
from pyngrok import ngrok
import os

# Ensure your token is set
NGROK_TOKEN = "3BMFHzAqnhcjvfBmlGFnGxm6ccl_55dcGpDSxS6fxGXcS9EiD"
ngrok.set_auth_token(NGROK_TOKEN)

# Force kill all existing ngrok processes to clear stuck tunnels
os.system("pkill ngrok")
ngrok.kill()

try:
    # Connect to port 8501
    public_url = ngrok.connect(8501)
    print("Streamlit App is live at:", public_url)

    # Run the streamlit app in the background - using the correct filename
    get_ipython().system_raw("streamlit run streamlit_app.py &")
    print("Streamlit process started in background.")
except Exception as e:
    print(f"Error: {e}")